# Evaluation Judge

Scores generated milestone snapshots with a resumable worker and rebuilds baseline and edited performance artifacts.

## Environment

In [ ]:
import sys
from datetime import datetime
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate.judge_utils import resolve_evaluation_selection

LOG_DIR = PROJECT_ROOT / "data" / "evaluation" / ".logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = None

def resolve_run_indices(n_runs: int, run_start: int = 1, run_count: int | None = None) -> list[int]:
    if n_runs <= 0 or run_start <= 0:
        raise ValueError("run settings must be positive")
    count = n_runs - run_start + 1 if run_count is None else int(run_count)
    if count <= 0 or run_start + count - 1 > n_runs:
        raise ValueError("requested run range is outside n_runs")
    return list(range(run_start, run_start + count))

## Evaluation Configuration

In [ ]:
import copy

from src.evaluate import load_selected_evaluation_configuration

experiment_config, selected_judge_config = load_selected_evaluation_configuration()
eval_type = experiment_config["eval_type"]
LOG_PATH = LOG_DIR / f"{eval_type}-judge-{datetime.now():%Y-%m-%d-%H-%M-%S}.log"
evaluation_dir = PROJECT_ROOT / "data" / "evaluation"
EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / f"{eval_type}.yaml"

editing_methods, eval_milestones, run_indices = resolve_evaluation_selection(
    experiment_config, eval_type
)
model_key = experiment_config["model_key"]
result_names = list(experiment_config["result_names"])
if eval_type == "intrinsic":
    study = experiment_config["studies"][experiment_config["study"]]
    n_runs = int(study.get("n_runs", experiment_config.get("n_runs", len(study["runs"]))))
else:
    n_runs = int(experiment_config["n_runs"])
judge_repeats = int(selected_judge_config["judge"]["repeats"])
base_seed = int(experiment_config["base_seed"])

judge_config = copy.deepcopy(selected_judge_config["judge"])
judge_config["log_path"] = LOG_PATH
judge_config.setdefault("model_config", {})["seed"] = base_seed
worker_config = copy.deepcopy(selected_judge_config["worker"])

## Manual Actions

In [ ]:
from src.evaluate.judge import (
    materialize_base_perf_results,
    score_pending_unscored,
    rebuild_perf_from_scored,
)

# Materialize base-model performance files if they do not already exist:
# materialize_base_perf_results(
#     model_key=model_key,
#     evaluation_dir=evaluation_dir,
#     eval_type=eval_type,
#     milestones=eval_milestones,
#     n_runs=n_runs,
#     judge_repeats=judge_repeats,
#     base_seed=base_seed,
#     result_names=result_names,
#     run_indices=run_indices,
#     log_path=LOG_PATH,
#     overwrite=False,
# )

# Score pending unscored files; this is resumable and can run independently:
# await score_pending_unscored(
#     model_key=model_key,
#     evaluation_dir=evaluation_dir,
#     eval_type=eval_type,
#     milestones=eval_milestones,
#     n_runs=n_runs,
#     judge_repeats=judge_repeats,
#     judge_config=judge_config,
#     methods=editing_methods,
#     result_names=result_names,
#     run_indices=run_indices,
#     **worker_config,
# )

# Rebuild edited-method performance files after scoring is complete:
# rebuild_perf_from_scored(
#     model_key=model_key,
#     evaluation_dir=evaluation_dir,
#     eval_type=eval_type,
#     milestones=eval_milestones,
#     n_runs=n_runs,
#     judge_repeats=judge_repeats,
#     methods=editing_methods,
#     result_names=result_names,
#     run_indices=run_indices,
#     log_path=LOG_PATH,
# )